In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  
import shap
import joblib
import json
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv('../data/processed/nba_final.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df = df.dropna(subset=['roll_pts_scored', 'roll_win_rate', 'team_elo'])


with open('../src/features.json', 'r') as f:
    FEATURES = json.load(f)


lr        = joblib.load('../src/logistic_regression.pkl')
rf        = joblib.load('../src/random_forest.pkl')
xgb_model = joblib.load('../src/xgboost_model.pkl')
scaler    = joblib.load('../src/scaler.pkl')


X = df[FEATURES]
y = df['result']
split_idx    = int(len(df) * 0.80)
X_train      = X.iloc[:split_idx]
X_test       = X.iloc[split_idx:]
y_train      = y.iloc[:split_idx]
y_test       = y.iloc[split_idx:]
X_test_scaled = scaler.transform(X_test)

print("Everything loaded!")
print(f"Test set: {len(X_test)} games")

Matplotlib is building the font cache; this may take a moment.


Everything loaded!
Test set: 6251 games


In [2]:
import os
os.makedirs('../assets', exist_ok=True)


feat_imp = pd.DataFrame({
    'feature':    FEATURES,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))

colors = ['#6B62C4' if i >= len(FEATURES)-3 
          else '#B8B4E8' for i in range(len(feat_imp))]

bars = ax.barh(feat_imp['feature'], feat_imp['importance'], color=colors)

ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('XGBoost Feature Importance\n(darker = top 3 features)', 
             fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


for bar, val in zip(bars, feat_imp['importance']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../assets/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to assets/feature_importance.png")

Saved to assets/feature_importance.png


In [3]:
print("Calculating SHAP values (takes 1-2 minutes)...")


explainer   = shap.TreeExplainer(xgb_model)


X_test_sample = X_test.iloc[:1000]
shap_values   = explainer.shap_values(X_test_sample)


shap.summary_plot(
    shap_values, 
    X_test_sample, 
    feature_names=FEATURES,
    show=False
)
plt.title('SHAP Feature Impact on Predictions', fontsize=14)
plt.tight_layout()
plt.savefig('../assets/shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved to assets/shap_summary.png")

Calculating SHAP values (takes 1-2 minutes)...
Saved to assets/shap_summary.png


In [4]:

game_idx = 0 
game     = X_test.iloc[game_idx]
actual   = y_test.iloc[game_idx]


prob = xgb_model.predict_proba(game.values.reshape(1,-1))[0][1]
pred = "WIN" if prob > 0.5 else "LOSS"

print(f"Game details:")
print(f"  Team ELO:       {game['team_elo']:.0f}")
print(f"  Opponent ELO:   {game['opp_elo']:.0f}")
print(f"  ELO difference: {game['elo_diff']:.0f}")
print(f"  Home game:      {'Yes' if game['is_home']==1 else 'No'}")
print(f"  Recent win rate:{game['roll_win_rate']:.2f}")
print(f"  Days rest:      {game['days_rest']:.0f}")
print(f"\nModel predicted: {pred} ({prob:.1%} confidence)")
print(f"Actual result:   {'WIN' if actual==1 else 'LOSS'}")
print(f"Model was:       {'CORRECT ✓' if pred==('WIN' if actual==1 else 'LOSS') else 'WRONG ✗'}")


shap_single = explainer.shap_values(game.values.reshape(1,-1))[0]

fig, ax = plt.subplots(figsize=(9, 6))
colors  = ['#6B62C4' if v > 0 else '#D97070' for v in shap_single]
y_pos   = range(len(FEATURES))

ax.barh(y_pos, shap_single, color=colors)
ax.set_yticks(y_pos)
ax.set_yticklabels(FEATURES)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('SHAP value (positive = pushes toward WIN)')
ax.set_title(f'Why did the model predict {pred}?\n'
             f'Confidence: {prob:.1%} | Actual: {"WIN" if actual==1 else "LOSS"}',
             fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../assets/shap_single_game.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to assets/shap_single_game.png")

Game details:
  Team ELO:       1447
  Opponent ELO:   1423
  ELO difference: 24
  Home game:      No
  Recent win rate:0.40
  Days rest:      1

Model predicted: LOSS (33.1% confidence)
Actual result:   LOSS
Model was:       CORRECT ✓
Saved to assets/shap_single_game.png


In [5]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(8, 6))


ax.plot([0,1], [0,1], 'k--', linewidth=1.5, label='Perfect calibration')


model_data = [
    ('Logistic Regression', 
     lr.predict_proba(X_test_scaled)[:,1], '#6B62C4'),
    ('Random Forest',       
     rf.predict_proba(X_test)[:,1],        '#2E8A72'),
    ('XGBoost',             
     xgb_model.predict_proba(X_test)[:,1], '#D97070'),
]

for name, probs, color in model_data:
    fraction_pos, mean_pred = calibration_curve(
        y_test, probs, n_bins=10
    )
    ax.plot(mean_pred, fraction_pos, 
            marker='o', linewidth=2,
            label=name, color=color)

ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Actual Win Rate',            fontsize=12)
ax.set_title('Calibration Curves\n'
             'Closer to dashed line = more trustworthy probabilities',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../assets/calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to assets/calibration.png")

Saved to assets/calibration.png


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

print("=" * 50)
print("        FINAL EVALUATION SUMMARY")
print("=" * 50)

models = [
    ('Logistic Regression', 
     lr.predict(X_test_scaled),
     lr.predict_proba(X_test_scaled)[:,1]),
    ('Random Forest',       
     rf.predict(X_test),
     rf.predict_proba(X_test)[:,1]),
    ('XGBoost',             
     xgb_model.predict(X_test),
     xgb_model.predict_proba(X_test)[:,1]),
]

for name, preds, probs in models:
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    print(f"\n{name}")
    print(f"  Accuracy : {acc:.3f} ({acc*100:.1f}%)")
    print(f"  ROC-AUC  : {auc:.3f}")

print("\n" + "=" * 50)
print(f"Baseline (home team always): ~58.0%")
print(f"Your improvement over baseline: +{(0.671-0.58)*100:.1f}%")
print("=" * 50)
print("\nCharts saved to assets/ folder")


        FINAL EVALUATION SUMMARY

Logistic Regression
  Accuracy : 0.671 (67.1%)
  ROC-AUC  : 0.737

Random Forest
  Accuracy : 0.670 (67.0%)
  ROC-AUC  : 0.734

XGBoost
  Accuracy : 0.673 (67.3%)
  ROC-AUC  : 0.732

Baseline (home team always): ~58.0%
Your improvement over baseline: +9.1%

Charts saved to assets/ folder
Ready for Day 6 — building the Gradio app!
